# Chicago Community Conditions, Crime & School Safety | SQL Analysis

## Project overview
This project uses SQL and Python to explore three prepared Chicago datasets: community census indicators, public-school records, and reported crime incidents. The analysis focuses on poverty and hardship, school safety, crime categories, and the geographic distribution of recorded incidents.

### Objectives
- Identify communities with low per-capita income or high poverty percentages.
- Explore crime categories, including incidents recorded at schools.
- Compare average safety scores across school types.
- Use SQL subqueries to connect crime counts and hardship measures with community area names.

**Tools:** Python, Pandas, SQLite, Jupyter Notebook, SQL magic.

**Data sources:** City of Chicago public datasets, distributed as prepared educational subsets by IBM Skills Network. The notebook's SQL queries and saved outputs are retained from the completed exercise. These records are a subset and are not a comprehensive or current citywide crime inventory.


### Data sources (downloadable CSV files)
- [Chicago Census Data](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv)
- [Chicago Public Schools](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv)
- [Chicago Crime Data](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv)

These are the prepared CSV subsets used by the notebook, not the complete source datasets. Original public data: City of Chicago; prepared educational subsets: IBM Skills Network.

## Data preparation
The code below creates a SQLite database (`FinalDB.db`) and loads three CSV files into `CENSUS_DATA`, `CHICAGO_PUBLIC_SCHOOLS`, and `CHICAGO_CRIME_DATA`. Execute the setup cells in order before running the SQL queries.


## Database setup

In [4]:
import csv, sqlite3
import pandas as pd

con = sqlite3.connect("FinalDB.db")
cur = con.cursor()



In [5]:
%load_ext sql


In [12]:
census_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv"
census_df = pd.read_csv(census_url)
census_df.to_sql("CENSUS_DATA", con, if_exists="replace", index=False)


print(census_df.shape)
census_df.head()

(78, 9)


,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


In [14]:

schools_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv"
schools_df = pd.read_csv(schools_url)
schools_df.to_sql("CHICAGO_PUBLIC_SCHOOLS", con, if_exists="replace", index=False)


print(schools_df.shape)
schools_df.head()

(566, 78)


,School_ID,NAME_OF_SCHOOL,"Elementary, Middle, or High School",Street_Address,City,State,ZIP_Code,Phone_Number,Link,Network_Manager,...,Freshman_on_Track_Rate__,X_COORDINATE,Y_COORDINATE,Latitude,Longitude,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,Ward,Police_District,Location
0,610038,Abraham Lincoln Elementary School,ES,615 W Kemper Pl,Chicago,IL,60614,(773) 534-5720,http://schoolreports.cps.edu/SchoolProgressRep...,Fullerton Elementary Network,...,NDA,1171699.458,1915829.428,41.924497,-87.644522,7,LINCOLN PARK,43,18,"(41.92449696, -87.64452163)"
1,610281,Adam Clayton Powell Paideia Community Academy ...,ES,7511 S South Shore Dr,Chicago,IL,60649,(773) 535-6650,http://schoolreports.cps.edu/SchoolProgressRep...,Skyway Elementary Network,...,NDA,1196129.985,1856209.466,41.760324,-87.556736,43,SOUTH SHORE,7,4,"(41.76032435, -87.55673627)"
2,610185,Adlai E Stevenson Elementary School,ES,8010 S Kostner Ave,Chicago,IL,60652,(773) 535-2280,http://schoolreports.cps.edu/SchoolProgressRep...,Midway Elementary Network,...,NDA,1148427.165,1851012.215,41.747111,-87.731702,70,ASHBURN,13,8,"(41.74711093, -87.73170248)"
3,609993,Agustin Lara Elementary Academy,ES,4619 S Wolcott Ave,Chicago,IL,60609,(773) 535-4389,http://schoolreports.cps.edu/SchoolProgressRep...,Pershing Elementary Network,...,NDA,1164504.290,1873959.199,41.809757,-87.672145,61,NEW CITY,20,9,"(41.8097569, -87.6721446)"
4,610513,Air Force Academy High School,HS,3630 S Wells St,Chicago,IL,60609,(773) 535-1590,http://schoolreports.cps.edu/SchoolProgressRep...,Southwest Side High School Network,...,91.8,1175177.622,1880745.126,41.828146,-87.632794,34,ARMOUR SQUARE,11,9,"(41.82814609, -87.63279369)"


In [16]:
crime_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv"
crime_df = pd.read_csv(crime_url)
crime_df.to_sql("CHICAGO_CRIME_DATA", con, if_exists="replace", index=False)


print(crime_df.shape)
crime_df.head()

(533, 21)


,ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,...,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
0,3512276,HK587712,2004-08-28,047XX S KEDZIE AVE,890,THEFT,FROM BUILDING,SMALL RETAIL STORE,False,False,...,9,14.0,58.0,6,1155838.0,1873050.0,2004,41.807440,-87.703956,"(41.8074405, -87.703955849)"
1,3406613,HK456306,2004-06-26,009XX N CENTRAL PARK AVE,820,THEFT,$500 AND UNDER,OTHER,False,False,...,11,27.0,23.0,6,1152206.0,1906127.0,2004,41.898280,-87.716406,"(41.898279962, -87.716405505)"
2,8002131,HT233595,2011-04-04,043XX S WABASH AVE,820,THEFT,$500 AND UNDER,NURSING HOME/RETIREMENT HOME,False,False,...,2,3.0,38.0,6,1177436.0,1876313.0,2011,41.815933,-87.624642,"(41.815933131, -87.624642127)"
3,7903289,HT133522,2010-12-30,083XX S KINGSTON AVE,840,THEFT,FINANCIAL ID THEFT: OVER $300,RESIDENCE,False,False,...,4,7.0,46.0,6,1194622.0,1850125.0,2010,41.743665,-87.562463,"(41.743665322, -87.562462756)"
4,10402076,HZ138551,2016-02-02,033XX W 66TH ST,820,THEFT,$500 AND UNDER,ALLEY,False,False,...,8,15.0,66.0,6,1155240.0,1860661.0,2016,41.773455,-87.706480,"(41.773455295, -87.706480471)"


In [17]:
%sql sqlite:///FinalDB.db

## SQL questions and results

The following queries retain the completed SQL analysis and its saved results.

### Analysis question 1

### Analysis question 1

##### Find the total number of crimes recorded in the CRIME table.


In [18]:
%%sql

SELECT COUNT(*) AS TOTAL_CRIMES
FROM CHICAGO_CRIME_DATA;

 * sqlite:///FinalDB.db
Done.


TOTAL_CRIMES
533


### Analysis question 2

##### List community area names and numbers with per capita income less than 11000.


In [19]:
%%sql

SELECT COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER
FROM CENSUS_DATA
WHERE PER_CAPITA_INCOME < 11000;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
West Garfield Park,26.0
South Lawndale,30.0
Fuller Park,37.0
Riverdale,54.0


### Analysis question 3

##### List all case numbers for crimes involving minors?(children are not considered minors for the purposes of crime analysis) 


In [20]:
%%sql

SELECT CASE_NUMBER
FROM CHICAGO_CRIME_DATA
WHERE DESCRIPTION LIKE '%MINOR%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER
HL266884
HK238408


### Analysis question 4

##### List all kidnapping crimes involving a child?


In [21]:
%%sql

SELECT *
FROM CHICAGO_CRIME_DATA
WHERE PRIMARY_TYPE = 'KIDNAPPING'
AND DESCRIPTION LIKE '%CHILD%';

 * sqlite:///FinalDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
5276766,HN144152,2007-01-26,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


### Analysis question 5

##### List the kind of crimes that were recorded at schools. (No repetitions)


In [22]:
%%sql

SELECT DISTINCT PRIMARY_TYPE
FROM CHICAGO_CRIME_DATA
WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%';

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


### Analysis question 6

##### List the type of schools along with the average safety score for each type.


In [23]:
%%sql

SELECT "Elementary, Middle, or High School",
       AVG(SAFETY_SCORE) AS AVERAGE_SAFETY_SCORE
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.


"Elementary, Middle, or High School",AVERAGE_SAFETY_SCORE
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


### Analysis question 7

##### List 5 community areas with highest % of households below poverty line


In [24]:
%%sql

SELECT COMMUNITY_AREA_NAME,
       PERCENT_HOUSEHOLDS_BELOW_POVERTY
FROM CENSUS_DATA
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


### Analysis question 8

##### Which community area is most crime prone? Display the coumminty area number only.


In [25]:
%%sql

SELECT COMMUNITY_AREA_NUMBER
FROM CHICAGO_CRIME_DATA
GROUP BY COMMUNITY_AREA_NUMBER
ORDER BY COUNT(*) DESC
LIMIT 1;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
25.0


### Analysis question 9

##### Use a sub-query to find the name of the community area with highest hardship index


In [26]:
%%sql

SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE HARDSHIP_INDEX = (
    SELECT MAX(HARDSHIP_INDEX)
    FROM CENSUS_DATA
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


### Analysis question 10

##### Use a sub-query to determine the Community Area Name with most number of crimes?


In [27]:
%%sql

SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_CRIME_DATA
    GROUP BY COMMUNITY_AREA_NUMBER
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Austin


## Key findings

The following results are supported by the saved SQL query outputs for the **provided subsets**:

- **Dataset coverage:** The loaded tables contain **78 census records**, **566 school records**, and **533 crime records**.
- **Low per-capita income:** **West Garfield Park, South Lawndale, Fuller Park, and Riverdale** are the four community areas returned by the query for per-capita income below **$11,000**.
- **Highest poverty percentages:** The five highest percentages of households below the poverty line are **Riverdale (56.5%)**, **Fuller Park (51.2%)**, **Englewood (46.6%)**, **North Lawndale (43.1%)**, and **East Garfield Park (42.4%)**.
- **School safety scores:** The average recorded safety scores are **49.52** for elementary schools (ES), **49.62** for high schools (HS), and **48.00** for middle schools (MS), excluding null values in the averages. These are descriptive averages, not evidence of a meaningful statistical difference.
- **Crime categories at schools:** Six distinct crime types appear in records with a school-related location description: **battery, criminal damage, narcotics, assault, criminal trespass, and public peace violation**.
- **Most recorded crimes in this subset:** Community area **25 (Austin)** has the highest number of records in the supplied crime subset. This identifies the highest **count**, not the highest population-adjusted crime rate.
- **Highest hardship index:** **Riverdale** is returned by the maximum-hardship-index query.

## Recommendations

1. **Prioritize further investigation of socioeconomic needs.** Review the communities with the highest reported poverty percentages using more recent census measures and relevant service-access data before recommending resource allocation.
2. **Explore school safety in greater depth.** Examine distributions, missing scores, school counts, and context by school type rather than relying only on group averages.
3. **Assess school-related crime with appropriate context.** Investigate the six recorded school-location crime categories using a fuller, time-aligned dataset; the subset cannot establish citywide prevalence.
4. **Use comparable crime measures.** For geographic comparisons, obtain complete incident counts, a defined reporting period, and population denominators before drawing conclusions about crime rates.

## Conclusion and key takeaways

This project shows how SQL can combine information from separate public datasets to answer focused questions about community socioeconomic conditions, school safety, and reported crimes. Filtering and sorting identified communities with low per-capita income or high poverty percentages; aggregation summarized safety scores by school type; and subqueries identified the community with the highest recorded hardship index and the community associated with the most incidents in the crime subset.

The results demonstrate that **community-level differences can be described using SQL, but they cannot be explained by these queries alone**. Austin's leading crime count is a property of the supplied 533-record subset and should not be described as the city's highest crime rate. Similarly, the school safety averages do not establish why scores differ between school types.

**Key takeaways**
- **SQL analysis:** Applied `WHERE`, `LIKE`, `DISTINCT`, `GROUP BY`, `AVG`, `COUNT`, `ORDER BY`, `LIMIT`, and nested queries to answer defined analytical questions.
- **Cross-dataset reasoning:** Connected community identifiers to community names and census measures to make query outputs more interpretable.
- **Data interpretation:** Distinguished incident counts from rates, group averages from statistically tested differences, and observed patterns from causal explanations.
- **Next steps:** A fuller, current, consistently timed dataset and appropriate denominators would be needed for stronger comparisons and actionable policy recommendations.

### Scope and limitations
The CSVs are educational subsets of public datasets. The saved results describe only those records, and different source datasets may cover different years. Findings should not be generalized to current conditions or interpreted as causal relationships.

### Attribution
Underlying data: City of Chicago public data. Prepared subsets and original instructional exercises: IBM Skills Network. The SQL analysis was completed in an educational context and adapted for this portfolio.
